# Synthetic Data Pipeline

This notebook runs the synthetic data generation pipeline.

### Before getting started:
- Ensure you have read `docs/*` and `README.md`
- Check that `params.py` and `config.py` are correct.
- Check the `call_LLM` and `red_write_data` functions in `processing.py` are correctly configured for your platform .
- Check that your input data exists and is correctly formatted.

First, import the required classes. 

In [1]:
import sys
from pathlib import Path
project_root = Path().resolve().parent
sys.path.append(str(project_root))

In [2]:
from src.data_generator import generate_patients, generate_admissions, generate_journeys, generate_clinical_notes, add_augmentations, save_final_outputs
from datetime import datetime
from config.params import PARAMS

/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:405: SyntaxWarning: invalid escape sequence '\s'
  df["ChiefComplaintDescription"] = df["ChiefComplaintDescription"].str.split("\s+\(").str.get(0)
/mnt/c/Users/Will Poulett/Documents/synthetic_clinical_notes/src/data_generator.py:406: SyntaxWarning: invalid escape sequence '\s'
  df["DiagnosisDescription"] = df["DiagnosisDescription"].str.split("\s+\(").str.get(0)


In [3]:
# INSERT RUN NAME BELOW
# This will be saved in the journey dataset at the end of the notebook for evaluation purposes
run_name = "wp_add_orthopaedic_letter"
current_time = datetime.now()
version_tag = current_time.strftime("%Y-%m-%d") + f"/{run_name}"
print(f"Run name: {run_name}\nDate: {current_time}\nVersion tag: {version_tag}")

Run name: wp_add_orthopaedic_letter
Date: 2026-05-06 12:40:27.268052
Version tag: 2026-05-06/wp_add_orthopaedic_letter


## 1 - Get Patients Information and Admissions

- Generates a list of patients with corresponding admission reasons.


In [6]:
patient_generator = generate_patients()
patients = await patient_generator.run(return_output = True)
patient_generator.write_patients_to_dataset()

Generating 1 patients... DONE


In [7]:
admission_generator = generate_admissions()
admissions = await admission_generator.run(return_output = True)
admission_generator.write_admissions_to_dataset()

Generating 1 admissions... DONE


In [8]:
admissions[0][0]

'{"date": "2026-01-04", "time": "18:50", "method": "A&E", "chief_complaint": "Right knee pain and swelling after twisting injury while stepping off a curb earlier today; difficulty weight-bearing.", "ED_diagnosis": "Suspected right medial meniscal injury", "triage_category": "Category 4 (Less urgent)", "allergies": "[]", "current_medications": "[\\"Lisinopril 10 mg once daily for hypertension\\", \\"Sertraline 50 mg once daily for depression\\", \\"Occasional ibuprofen 400 mg as needed for musculoskeletal pain (no dose taken today)\\"]", "past_medical_history": "[\\"Primary hypertension diagnosed 2016, well controlled\\", \\"Generalised anxiety and depression, stable on SSRI\\", \\"BMI 29 (overweight)\\", \\"No previous knee injuries or surgery\\", \\"No history of thromboembolic disease\\"]", "admitting_consultant": "Dr. Gary Albert Perkins (Consultant)", "ward": "Acute Orthopaedic Assessment Unit", "specialty": "Trauma and Orthopaedics", "admission_type": "emergency", "surgery_requir

## 2 - Generating and Filtering Journeys

- Generates, validates, and adds details to patient journeys.
- Filtering removes any document types that are not listed as possible event types in `params.py`.

In [9]:
journey_generator = generate_journeys()
journeys = await journey_generator.run(return_outputs = True)
journey_generator.write_journeys_to_dataset()

Generating simple journeys... Validating simple journeys...
Validator Changes:
 {'Patient_0': 0}
Generating extra details... Generating staff personas... Creating full detailed journeys... Filtering journeys...
Patient 0 - Removing events: []
Journey 0 has length 21
DONE


In [10]:
journeys[0]

['{"event_type": "ED event", "date": "2026-01-04", "time": "18:50", "staff": "[\\"Nurse Sharon Vanessa Adams\\"]", "details": "On arrival to ED at 18:50, Nurse Sharon Vanessa Adams completes initial streaming, notes marked swelling of the right knee with difficulty weight-bearing, and escorts the patient to the minor injuries waiting area in a wheelchair. No medications are given at this stage and no tests are performed.", "next_steps_decision": "Patient to be booked for formal ED triage assessment and analgesia review by nursing staff as soon as a cubicle is available."}',
 '{"event_type": "ED event", "date": "2026-01-04", "time": "19:10", "staff": "[\\"Nurse Sharon Vanessa Adams (ED Nurse)\\"]", "details": "Nurse Sharon Vanessa Adams completes triage observations, notes right knee effusion with reduced active flexion due to pain, and administers oral paracetamol 1 g and ibuprofen 400 mg as first-line analgesia before providing elbow crutches and instructing the patient to remain non-

In [5]:
import json
json.loads(journeys[0][15])

NameError: name 'journeys' is not defined

## 3 - Generate and Validate Clinical Notes

- Uses LLMs to generate clinical notes. 
- Validates each note using an LLM Judge. 

In [4]:
clinical_note_generator = generate_clinical_notes()
notes = await clinical_note_generator.run(return_output = True)
clinical_note_generator.write_patient_documents_to_dataset()

Patient 0: Generating Notes... Validating Notes... Estimated 14 changes... Combining sections... {'note_subject': 'ED streaming', 'note_type': 'ED', 'Content': 'Date: 04/01/26\nTime: 18:50\nLocation: ED streaming area / minor injuries waiting area\nClinician: Nurse Sharon Vanessa Adams\n\nPatient: Nora Denise Stephenson\nDOB: 14/03/1978 (Age 47)\nGender: Female\nNHS number: 576688058\nMedical record number: 198013872\n\nPresenting complaint: Right knee pain and swelling after twisting injury while stepping off a curb earlier today; difficulty weight-bearing.\n\nCurrent medications (per records): Lisinopril 10 mg once daily for hypertension; Sertraline 50 mg once daily for depression; occasional ibuprofen 400 mg as needed for musculoskeletal pain (no dose taken today).\nAllergies: No known drug allergies reported.\n\nInitial streaming event: On arrival to ED at 18:50 I complete initial streaming, note marked swelling of the right knee with difficulty weight-bearing, and escort the patie

In [12]:
import json
print(json.loads(notes[0][13])["Content"])

05/01/26

To: Orthopaedic Knee Clinic

Re: Nora Denise Stephenson
DOB: 14/03/78   Age: 47 years
NHS No: 576688058
Hospital No: 198013872

Dear Colleague,

I would be grateful if you could review Mrs Nora Stephenson in your outpatient knee clinic for further assessment and management of a suspected right medial meniscal tear.

Mrs Stephenson was admitted via A&E on 04/01/26 following a twisting injury to her right knee while stepping off a kerb earlier that day. She experienced immediate pain and swelling with difficulty weight-bearing. She reports pain predominantly over the medial aspect of the knee. There has been no true locking, no giving way, and no systemic symptoms.

Relevant background includes primary hypertension (diagnosed 2016, well controlled) and generalised anxiety and depression, stable on sertraline. She has a BMI of 29. There is no history of previous knee injury or surgery and no history of thromboembolic disease. She reports no drug allergies.

Usual medications are

## 4 - Add Augmentations to clinical Notes

This section allows for the augmentatio of clinical notes by:

- Replacing long phrases with abbreviations.
- Adding typos.
- Adding signatures.

In [ ]:
augmentator = add_augmentations()
augmented_notes = await augmentator.run(True)
augmentator.write_final_documents_to_dataset()

## 5 - Write Clinical Notes to Dataset

- Writes the clinical notes to a dataset, alongside the patient journey and admission details.

In [ ]:
output_saver = save_final_outputs()
output_saver.run(run_name, current_time, version_tag)